# 02 — Connecting to MLflow

MLflow tracks your machine learning experiments — it records parameters, metrics, and results so you can compare runs over time.

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Install MLflow

In [ ]:
!uv pip install mlflow -q

## Step 2 — Get an auth token

MLflow on OpenShift requires a bearer token. The cell below tries the mounted ServiceAccount token first (automatically available inside a notebook pod), then falls back to `oc whoami -t`.

In [ ]:
import os
import subprocess

SA_TOKEN_FILE = "/var/run/secrets/kubernetes.io/serviceaccount/token"

if os.path.isfile(SA_TOKEN_FILE):
    with open(SA_TOKEN_FILE) as f:
        TOKEN = f.read().strip()
    print("Using mounted ServiceAccount token")
else:
    TOKEN = subprocess.run(
        ["oc", "whoami", "-t"], capture_output=True, text=True
    ).stdout.strip()
    print("Using oc whoami -t token")

os.environ["MLFLOW_TRACKING_TOKEN"] = TOKEN
print("Token set:", TOKEN[:20], "...")

## Step 3 — Set your MLflow tracking URL

Replace the URL below with the MLflow server URL provided by your instructor.

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "http://mlflow:5000"   # update this URL

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("my-first-experiment")

print("Tracking URI:", mlflow.get_tracking_uri())

## Step 4 — Log a simple run

This records one experiment run with a parameter and a metric.

In [ ]:
with mlflow.start_run() as run:
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_metric("accuracy", 0.95)

    print("Run ID:", run.info.run_id)
    print("Done! Open the MLflow UI to see your run.")

## Step 5 — (Optional) Auto-trace LangChain calls

If you are building an AI agent with LangChain or LangGraph, one line automatically records every LLM call and tool execution in MLflow — no extra code needed.

In [ ]:
mlflow.langchain.autolog()

print("LangChain autologging enabled.")
print("Any LangChain / LangGraph calls made after this point will be traced in MLflow.")